<a href="https://colab.research.google.com/github/HassanDastagir/Solar-Roof-Segmentation-Project/blob/main/Solar_Roof_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# final============================================================
# IMPORTS
# ============================================================
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from skimage import measure, morphology
from skimage.draw import polygon
from skimage.transform import resize
import ipywidgets as widgets
from ipywidgets import interactive
from google.colab import drive
from PIL import Image
import requests
from io import BytesIO
import math

# ============================================================
# MOUNT DRIVE
# ============================================================
drive.mount('/content/drive')

# ============================================================
# USER PARAMETERS
# ============================================================
LATITUDE   = 51.085069029513
ZOOM_LEVEL = 20
SCALE      = 2

PANEL_WIDTH_M  = 1.1
PANEL_HEIGHT_M = 2.1
PANEL_FILL_THRESHOLD = 0.90

MODEL_SIZE = 768
SQFT_PER_SQM = 10.7639

# ============================================================
# PATHS
# ============================================================
IMAGE_PATH = "/content/drive/MyDrive/Version Two/Canada /H - M live folder/H - M - Test 4.png"
MODEL_FULL_PATH  = "/content/drive/MyDrive/Version Two/Canada /Second 281 Pictures Batch/best_model_full (1).keras"
MODEL_OPT_PATH   = "/content/drive/MyDrive/Version Two/Canada /Second 281 Pictures Batch/optimal_best_model.keras"
PANEL_IMAGE_PATH = "/content/drive/MyDrive/solar1.png"

# ============================================================
# LOAD SOLAR PANEL IMAGE
# ============================================================
panel_img = cv2.imread(PANEL_IMAGE_PATH, cv2.IMREAD_UNCHANGED)

if panel_img.shape[2] == 3:
    panel_img = cv2.cvtColor(panel_img, cv2.COLOR_BGR2RGBA)
else:
    panel_img = cv2.cvtColor(panel_img, cv2.COLOR_BGRA2RGBA)

# ============================================================
# LOAD IMAGE
# ============================================================
def load_image(path):
    if path.startswith("http"):
        response = requests.get(path)
        img = Image.open(BytesIO(response.content))
    else:
        img = Image.open(path)
    return np.array(img.convert("RGB"))

image_orig = load_image(IMAGE_PATH)
H_orig, W_orig = image_orig.shape[:2]

# ============================================================
# PREPARE IMAGE FOR MODEL
# ============================================================
image_model = cv2.resize(image_orig, (MODEL_SIZE, MODEL_SIZE))
image_norm  = image_model.astype(np.float32) / 255.0

# ============================================================
# LOAD MODELS
# ============================================================
model_full  = tf.keras.models.load_model(MODEL_FULL_PATH)
model_opt   = tf.keras.models.load_model(MODEL_OPT_PATH)

# ============================================================
# PREDICTIONS
# ============================================================
pred_full  = model_full.predict(image_norm[None], verbose=0)[0]
pred_opt   = model_opt.predict(image_norm[None], verbose=0)[0]

mask_full  = (pred_full > 0.5).astype(np.uint8)
mask_opt   = (pred_opt  > 0.5).astype(np.uint8)

# ============================================================
# CLEAN MASKS
# ============================================================
def clean_mask(mask, min_size=500):
    kernel = np.ones((3,3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = morphology.remove_small_objects(mask.astype(bool), min_size)
    return mask.astype(np.uint8)

mask_full = clean_mask(mask_full)
mask_opt  = clean_mask(mask_opt)

# ============================================================
# UPSCALE MASKS
# ============================================================
def upscale_mask(mask):
    return resize(mask, (H_orig, W_orig),
                  order=0, preserve_range=True,
                  anti_aliasing=False).astype(np.uint8)

mask_full_orig  = upscale_mask(mask_full)
mask_opt_orig   = upscale_mask(mask_opt)

# ============================================================
# FIND ROOF CONTOURS
# ============================================================
contours = measure.find_contours(mask_full_orig, 0.5)

# ============================================================
# METERS PER PIXEL
# ============================================================
def meters_per_pixel(lat, zoom, scale):
    return (156543.03392 * math.cos(math.radians(lat))) / (2 ** zoom * scale)

MPP = meters_per_pixel(LATITUDE, ZOOM_LEVEL, SCALE)

PANEL_W_PX = max(1, int(PANEL_WIDTH_M / MPP))
PANEL_H_PX = max(1, int(PANEL_HEIGHT_M / MPP))

# ============================================================
# PANEL PLACEMENT
# ============================================================
def place_panels(opt_mask):
    H, W = opt_mask.shape
    covered = np.zeros_like(opt_mask, dtype=np.uint8)
    panels = []

    for y in range(0, H - PANEL_H_PX + 1, PANEL_H_PX):
        for x in range(0, W - PANEL_W_PX + 1, PANEL_W_PX):
            patch = opt_mask[y:y+PANEL_H_PX, x:x+PANEL_W_PX]
            if np.mean(patch) >= PANEL_FILL_THRESHOLD:
                uncovered = patch & (covered[y:y+PANEL_H_PX, x:x+PANEL_W_PX] == 0)
                if np.any(uncovered):
                    panels.append((x, y))
                    covered[y:y+PANEL_H_PX, x:x+PANEL_W_PX] |= patch

    covered &= opt_mask
    return panels, covered

# ============================================================
# AREA CALCULATION
# ============================================================
def calculate_area_sqft(mask):
    return np.sum(mask) * (MPP**2) * SQFT_PER_SQM

# ============================================================
# INTERACTIVE VISUALIZATION
# ============================================================
def update_plot(roof_id, panel_angle):

    fig, axes = plt.subplots(1, 2, figsize=(16,8))
    ax_zoom = axes[0]
    ax_full = axes[1]

    contour = contours[roof_id]

    roof_mask = np.zeros(mask_full_orig.shape, dtype=bool)
    rr, cc = polygon(contour[:,0].astype(int),
                     contour[:,1].astype(int),
                     roof_mask.shape)
    roof_mask[rr, cc] = True

    optimal_selected = mask_opt_orig & roof_mask
    panels, coverage = place_panels(optimal_selected)

    # ---------------- ZOOM VIEW ----------------
    ax_zoom.imshow(image_orig)
    ax_zoom.imshow(optimal_selected, cmap="Greens", alpha=0.45)
    ax_zoom.plot(contour[:,1], contour[:,0], 'white', linewidth=3)

    for oc in measure.find_contours(optimal_selected.astype(np.uint8), 0.5):
        ax_zoom.plot(oc[:,1], oc[:,0], color="white", linewidth=2)

    # -------- REAL SOLAR PANEL OVERLAY --------
    for (x, y) in panels:

        panel_resized = cv2.resize(panel_img, (PANEL_W_PX, PANEL_H_PX))

        center = (PANEL_W_PX / 2, PANEL_H_PX / 2)
        M = cv2.getRotationMatrix2D(center, panel_angle, 1.0)

        rotated = cv2.warpAffine(
            panel_resized,
            M,
            (PANEL_W_PX, PANEL_H_PX),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_TRANSPARENT
        )

        panel_rgb = rotated[:, :, :3]
        panel_alpha = rotated[:, :, 3] / 255.0

        ax_zoom.imshow(
            panel_rgb,
            extent=(x, x + PANEL_W_PX, y + PANEL_H_PX, y),
            alpha=panel_alpha
        )

    # Zoom crop
    ys, xs = np.where(roof_mask)
    y_min, y_max = ys.min(), ys.max()
    x_min, x_max = xs.min(), xs.max()

    padding = 40
    ax_zoom.set_xlim(max(0,x_min-padding), min(W_orig,x_max+padding))
    ax_zoom.set_ylim(min(H_orig,y_max+padding), max(0,y_min-padding))

    # ---------------- FULL IMAGE VIEW ----------------
    ax_full.imshow(image_orig)
    for i, contour_all in enumerate(contours):
        color = "yellow" if i == roof_id else "red"
        linewidth = 3 if i == roof_id else 1.5
        ax_full.plot(contour_all[:,1], contour_all[:,0],
                     color=color, linewidth=linewidth)

    ax_full.set_title("Full Image - All Detected Roofs")
    ax_full.axis("off")

    # ---------------- CALCULATIONS ----------------
    roof_area_sqft    = calculate_area_sqft(roof_mask)
    optimal_area_sqft = calculate_area_sqft(optimal_selected)
    panel_area_sqft   = calculate_area_sqft(coverage)

    roof_mask_model = resize(roof_mask, (MODEL_SIZE, MODEL_SIZE),
                             order=0, preserve_range=True,
                             anti_aliasing=False).astype(bool)

    optimal_model = resize(optimal_selected, (MODEL_SIZE, MODEL_SIZE),
                           order=0, preserve_range=True,
                           anti_aliasing=False).astype(np.uint8)

    roof_confidence = np.mean(pred_full[roof_mask_model])
    optimal_confidence = np.mean(pred_opt[optimal_model==1]) if np.any(optimal_model) else 0
    panel_fit_confidence = np.mean(coverage[optimal_selected==1]) if np.any(optimal_selected) else 0

    final_confidence = (
        0.5 * roof_confidence +
        0.3 * optimal_confidence +
        0.2 * panel_fit_confidence
    ) * 100

    ax_zoom.set_title(
        f"Roof {roof_id}\n"
        f"Roof: {roof_area_sqft:.1f} sq ft | "
        f"Optimal: {optimal_area_sqft:.1f} sq ft\n"
        f"Panels: {len(panels)} | "
        f"Panel Area: {panel_area_sqft:.1f} sq ft\n"
        f"Confidence: {final_confidence:.1f}% | "
        f"Angle: {panel_angle}°"
    )

    plt.tight_layout()
    plt.show()

    print("Roof ID:", roof_id)
    print(f"Roof Area: {roof_area_sqft:.2f} sq ft")
    print(f"Optimal Area: {optimal_area_sqft:.2f} sq ft")
    print(f"Panels Placed: {len(panels)}")
    print(f"Panel Covered Area: {panel_area_sqft:.2f} sq ft")
    print(f"Overall Confidence: {final_confidence:.2f}%")

# ============================================================
# DROPDOWN + SLIDER
# ============================================================
display(
    interactive(
        update_plot,
        roof_id=widgets.Dropdown(
            options=list(range(len(contours))),
            description="Roof ID"
        ),
        panel_angle=widgets.IntSlider(
            value=0,
            min=0,
            max=360,
            step=5,
            description="Panel Angle (°)"
        )
    )
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


interactive(children=(Dropdown(description='Roof ID', options=(0, 1, 2, 3), value=0), IntSlider(value=0, descr…